# 三角関数のユースケース集 — 現代社会の裏側を覗く

**このノートブックの内容**:
1. 音波の合成 (ドの音、和音)
2. GPS の三角測量
3. キャラクターの円運動 (ゲーム・アニメ)
4. フーリエ変換 (音色の正体)

各セルを Shift+Enter で実行してください。

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (03_trigonometry.md)](../03_trigonometry.md)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

## ユースケース 1: 音波

「ドの音」は周波数 約 261.63 Hz の sin の波。Python で作って描画してみる。

ラの音 (A4 = 440 Hz):

$$
y(t) = \sin(2\pi \cdot 440 \cdot t)
$$

In [ ]:
SAMPLE_RATE = 8000  # サンプリングレート (Hz)
DURATION = 0.05  # 50ms 分だけ描画 (短く)

# ドレミファソラシド (Hz)
notes = {
    'ド (C4)': 261.63,
    'レ (D4)': 293.66,
    'ミ (E4)': 329.63,
    'ファ (F4)': 349.23,
    'ソ (G4)': 392.00,
    'ラ (A4)': 440.00,
    'シ (B4)': 493.88,
    'ド (C5)': 523.25,
}

t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION))

fig, ax = plt.subplots(figsize=(12, 6))
for name, freq in notes.items():
    wave = np.sin(2 * np.pi * freq * t)
    ax.plot(t * 1000, wave + list(notes).index(name) * 2.5, label=f'{name}: {freq} Hz')

ax.set_xlabel('時間 (ミリ秒)')
ax.set_yticks([])
ax.set_title('音階の正体: 周波数だけが違う sin の波')
ax.legend(loc='right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 和音 (C メジャー = ド + ミ + ソ)

和音は、複数の sin 波を「足し算」したもの。

In [ ]:
do = np.sin(2 * np.pi * 261.63 * t)
mi = np.sin(2 * np.pi * 329.63 * t)
so = np.sin(2 * np.pi * 392.00 * t)
chord = do + mi + so   # Cメジャー

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t * 1000, chord, 'r-')
ax.set_xlabel('時間 (ミリ秒)')
ax.set_ylabel('振幅')
ax.set_title('C メジャー = ド + ミ + ソ の合成波')
ax.grid(True, alpha=0.3)
plt.show()

→ 単純な sin が3つ重なるだけで「和音」になる。**Bach の合唱曲も、本質はこれと同じ重ね合わせ** です。

## ユースケース 2: GPS の三角測量

GPS衛星3個からの距離が分かれば、地上のあなたの位置が決まる。
簡単のため、2次元で考えます (3つの円の交点を求める)。

3 つの衛星からの距離:

$$
d_i = \sqrt{(x - x_i)^2 + (y - y_i)^2}
$$

3 つの円の交点が現在位置

In [ ]:
def gps_demo() -> None:
    """3つのGPS衛星と、その距離円の交点でユーザー位置を可視化."""
    # 衛星の位置 (2次元簡略化)
    sat1 = np.array([0.0, 5.0])
    sat2 = np.array([5.0, 0.0])
    sat3 = np.array([5.0, 5.0])
    # ユーザーの真の位置
    user = np.array([2.5, 2.5])
    # 各衛星からの距離 (実際はGPSが時刻差から計算する)
    d1 = np.linalg.norm(sat1 - user)
    d2 = np.linalg.norm(sat2 - user)
    d3 = np.linalg.norm(sat3 - user)

    fig, ax = plt.subplots(figsize=(9, 9))
    for sat, d, color, name in zip(
        [sat1, sat2, sat3],
        [d1, d2, d3],
        ['blue', 'green', 'orange'],
        ['衛星1', '衛星2', '衛星3'],
    ):
        circle = plt.Circle(sat, d, fill=False, color=color, linewidth=2, label=f'{name} (距離 {d:.2f})')
        ax.add_patch(circle)
        ax.plot(*sat, marker='*', color=color, markersize=20)
        ax.text(sat[0] + 0.1, sat[1] + 0.1, name, fontsize=11)

    ax.plot(*user, 'r.', markersize=20, label=f'ユーザー位置 {tuple(user)}')
    ax.set_xlim(-6, 11)
    ax.set_ylim(-6, 11)
    ax.set_aspect('equal')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_title('GPS の原理: 3つの衛星からの距離円の交点')
    plt.show()

gps_demo()

3つの円が交わる1点が、あなたの位置。実際の GPS は4個目の衛星も使って **時刻誤差** も補正します。

3次元の場合は、距離の式が 
`distance = √((x−sx)² + (y−sy)² + (z−sz)²)`
になるだけ。**まさにピタゴラスの定理の拡張** です。

## ユースケース 3: キャラクターの円運動

ゲームやアニメでキャラクターを円運動させるには、時間 `t` で角度を進めて `(cos t, sin t)` を位置にする。

$$
(x, y) = (r \cos t, r \sin t)
$$

In [ ]:
def draw_circular_motion(t_value: float) -> None:
    """円運動の軌跡と現在位置."""
    radius = 1.0
    # 軌跡
    ts = np.linspace(0, 2 * np.pi, 100)
    xs = radius * np.cos(ts)
    ys = radius * np.sin(ts)
    # 現在位置
    x = radius * math.cos(t_value)
    y = radius * math.sin(t_value)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(xs, ys, 'gray', linewidth=1, alpha=0.5)
    ax.plot(x, y, 'ro', markersize=20)
    ax.text(x + 0.1, y + 0.1, '🐢', fontsize=22)
    ax.plot([0, x], [0, y], 'r--')
    ax.text(0, 0, '⊕', ha='center', va='center', fontsize=18)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f't = {t_value:.2f} rad → 位置 ({x:.3f}, {y:.3f})')
    plt.show()

interact(
    draw_circular_motion,
    t_value=FloatSlider(min=0, max=2 * math.pi, step=0.1, value=1.0, description='時間 t'),
);

## ユースケース 4: フーリエ変換 — 音色の正体

「**どんな音も、純粋な sin の波の足し算で表せる**」というのがフーリエの大発見 (1822年)。

実例: ノコギリ波 (saw wave) を sin の足し算で近似してみる。

フーリエ級数: 任意の周期関数は sin/cos の和で表現可能

$$
f(t) = \sum_{n=0}^{\infty} \bigl(a_n \cos(n\omega t) + b_n \sin(n\omega t)\bigr)
$$

In [ ]:
def fourier_sawtooth(n_terms: int) -> None:
    """ノコギリ波を n 項の sin で近似.

    ノコギリ波 = Σ (1/k) sin(k × t)  (k=1, 2, 3, ...)
    """
    t = np.linspace(0, 4 * np.pi, 1000)
    approx = np.zeros_like(t)
    for k in range(1, n_terms + 1):
        approx += (1 / k) * np.sin(k * t)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t, approx, 'r-', linewidth=2, label=f'{n_terms} 項の sin の合成')
    ax.set_xlabel('時間')
    ax.set_ylabel('振幅')
    ax.set_title(f'ノコギリ波の近似 (項数 = {n_terms})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

interact(
    fourier_sawtooth,
    n_terms=IntSlider(min=1, max=50, step=1, value=1, description='項数'),
);

**スライダーを動かしてみてください**: 
- 1項のときは普通の sin。
- 項数を増やすほどノコギリ状に。
- これが **「楽器の音色の違い」** の本質 — それぞれの楽器は、独自の倍音の組み合わせ。

### この技術の応用先
- **MP3 / AAC** などの音声圧縮
- **JPEG** の画像圧縮 (離散コサイン変換)
- **MRI**, **CT** スキャンの画像再構成
- **株価のサイクル分析**
- **ノイズキャンセリング** イヤホン
- **AI の Transformer の Positional Encoding**

**1822年のフランスの数学者の発見が、2026年の私たちの暮らしを支えています**。

## まとめ

今日見たもの:
1. 音 ← sin の波
2. GPS ← 距離 = ピタゴラスの定理の拡張、衛星の角度 = 三角法
3. キャラ動作 ← (cos t, sin t) の円運動
4. フーリエ ← どんな波でも sin の足し算で表せる

**三角関数は、世界を「波」として記述するための、最強の道具**です。

次は本格的な記号の学習へ → [`../../00_notation/README.md`](../../00_notation/README.md)

AI との接続を深めたい方は → [`../columns/05_math_in_ai.md`](../columns/05_math_in_ai.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`02_trigonometry.ipynb`](02_trigonometry.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [次の章: 00_notation](../../00_notation/README.md) |